# 입찰메이트 RAG — 서빙 E2E 평가 (KURE+Phi FT, 단일 컬렉션 정리본)

코랩 기준. **새 `retrieval.py`(듀얼 라우터 내장본)** 를 그대로 로드해 쓴다. 패치 셀이 없는 이유:
- `_build_chroma_where` 의 기관키 매핑은 이제 `BidMateRetriever(agency_meta_key=...)` 인스턴스 인자로 처리
- D타입 거절은 클래스 내장 `sigmoid(logit) < sig_th` (기본 0.5)
- HWP 컨텍스트의 `original_name` 누락은 `_normalize_chunk`/`_get_hwp_context` 에서 `source_file` 로 자동 alias

`[1]` 의 토글로 **kh_v3 ↔ chunks_all** 단일 평가를 전환한다. (둘을 type별로 섞어 돌리는 건 통합본을 사용)

개선점(이전 허접본 대비): 0점방지 EXPECT_MIN 검증 · question 기준 done(`id` 중복 회피) · `retrieved_names=source_file` · 디버그 셀 제거 · 출력 파일명에 태그 분리.


In [ ]:
# [0] 설치
!pip install -q chromadb sentence-transformers rank_bm25 kiwipiepy peft transformers accelerate openai tqdm nest_asyncio rapidfuzz
!pip uninstall -y torchao
print("설치 완료 — 재시작 메시지 뜨면 재시작 후 [1]부터")

In [ ]:
# [0b] 진단 — chroma 안의 컬렉션 이름·개수 + 메타 기관키 자동 확인 (마운트 후 1회)
import os, gc, chromadb
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass
_AGENCY=['agency','organization_cleaned','organization','agency_name','institution']
def _detect(metas):
    for k in _AGENCY:
        if any(isinstance(m,dict) and m.get(k) for m in metas): return k
    for m in metas:
        if isinstance(m,dict):
            for k,v in m.items():
                if isinstance(v,str) and v.strip(): return k
    return None
def list_cols(d):
    if not os.path.isdir(d): print('  (폴더 없음)', d); return
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    cl=chromadb.PersistentClient(path=d)
    print('  경로:', d)
    for c in cl.list_collections():
        try:
            col=cl.get_collection(c.name); cnt=col.count()
            s=col.get(limit=20, include=['metadatas'])['metadatas'] or []
            print(f'  - {c.name:34} count={cnt:>8,} | 기관키={_detect(s)} | 메타키={list(s[0].keys())[:8] if s else []}')
        except Exception as e: print(f'  - {c.name:34} (실패 {e})')
# 코랩에서 chroma 가 풀려있는 폴더들 점검 (실제 경로에 맞게 추가)
for p in ['/content/bidmate_kh_v3/chroma_db','/content/bidmate_chunks_all/chroma_db','/content/chroma_db']:
    print('▶', p); list_cols(p)
print('\n※ count>0 인 이름 → [1] COLLECTION_NAME / 감지 기관키 → [1] AGENCY_KEY')

In [ ]:
# [1] 마운트 + chroma 로컬 폴더 준비 (★ 0점 방지: EXPECT_MIN 미달이면 중단)
#  ┌─────────────────────────────────────────────────────────────────┐
#  │ 청킹 전환 스위치 — 이 블록만 토글하면 kh_v3 ↔ chunks_all 전환.       │
#  └─────────────────────────────────────────────────────────────────┘
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, gc, chromadb, json
DRIVE='/content/drive/MyDrive/data/bidmate'

# ===== kh_v3 (기본) =================================================
CHUNK_TAG       = 'kh_v3'
CHUNK_FILE      = 'chunks/kh_v3.json'
BM25_FILE       = 'bm25/bm25_index_bidmate_kh_v3_A-1.pkl'
CHROMA_DIR      = '/content/bidmate_kh_v3/chroma_db'
COLLECTION_NAME = 'bidmate_kh_v3_A-1'
EXPECT_MIN      = 35000
AGENCY_KEY      = 'auto'        # 자동 감지 (또는 'organization_cleaned' 등 직접 지정)

# # ----- chunks_all 로 돌릴 때: 위 블록 주석 처리하고 아래 사용 -----
# CHUNK_TAG       = 'chunks_all'
# CHUNK_FILE      = 'chunks/chunks_all.json'
# BM25_FILE       = 'bm25/bm25_index_bidmate_chunks_all_A-2.pkl'
# CHROMA_DIR      = '/content/bidmate_chunks_all/chroma_db'
# COLLECTION_NAME = 'bidmate_chunks_all'
# EXPECT_MIN      = 8000
# AGENCY_KEY      = 'agency'
# ===================================================================

LOCAL=f'/content/bidmate_{CHUNK_TAG}'
COLLECTION_NAME=COLLECTION_NAME.strip()
assert os.path.isdir(CHROMA_DIR), f'chroma 폴더 없음: {CHROMA_DIR} — [0b] 로 실제 경로 확인'

gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
_cl=chromadb.PersistentClient(path=CHROMA_DIR)
_names=[c.name for c in _cl.list_collections()]
assert COLLECTION_NAME in _names, f'컬렉션 {COLLECTION_NAME} 없음. 존재: {_names}'
_col=_cl.get_collection(COLLECTION_NAME)
EXPECT_N=_col.count()
assert EXPECT_N>=EXPECT_MIN, f'❌ {COLLECTION_NAME} count={EXPECT_N:,} < EXPECT_MIN({EXPECT_MIN:,}) — 빈/오염 컬렉션. [0b] 확인'

# 기관키 자동 감지
_AGENCY=['agency','organization_cleaned','organization','agency_name','institution']
if AGENCY_KEY=='auto':
    metas=_col.get(limit=30, include=['metadatas'])['metadatas'] or []
    AGENCY_KEY=next((k for k in _AGENCY if any(isinstance(m,dict) and m.get(k) for m in metas)), None)
    if AGENCY_KEY is None:
        for m in metas:
            if isinstance(m,dict):
                AGENCY_KEY=next((k for k,v in m.items() if isinstance(v,str) and v.strip()), None)
                if AGENCY_KEY: break
    assert AGENCY_KEY, '❌ 기관키 자동감지 실패 — [0b] 확인 후 직접 지정'
print(f'[{CHUNK_TAG}] 컬렉션={COLLECTION_NAME} | count={EXPECT_N:,} | 기관키={AGENCY_KEY!r} | 그외={_names}')

# 청크/bm25/eval 로컬 복사
for rel in [CHUNK_FILE, BM25_FILE, 'eval/eval_retrieval_579.csv']:
    s=f'{DRIVE}/{rel}'; d=f'{LOCAL}/{rel}'
    assert os.path.exists(s), f'원본 없음: {s}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d): shutil.copy(s, d)

FIXED='/content/bidmate'
os.makedirs(f'{FIXED}/eval', exist_ok=True)
shutil.copy(f'{LOCAL}/eval/eval_retrieval_579.csv', f'{FIXED}/eval/eval_retrieval_579.csv')
OUT_TAG_DIR=f'{FIXED}/outputs/{CHUNK_TAG}'   # ★ 태그별 분리 (kh_v3/chunks_all 결과 덮어쓰기 방지)
os.makedirs(OUT_TAG_DIR, exist_ok=True)
print('eval 동기화 / 출력폴더:', OUT_TAG_DIR)
_CHUNK_TAG,_CHUNK_FILE,_BM25_FILE,_COLLECTION_NAME,_EXPECT_N,_CHROMA_DIR,_OUT_TAG_DIR,_AGENCY_KEY = \
    CHUNK_TAG,CHUNK_FILE,BM25_FILE,COLLECTION_NAME,EXPECT_N,CHROMA_DIR,OUT_TAG_DIR,AGENCY_KEY

In [ ]:
# [2] config 주입
import sys, types, os
from pathlib import Path
CODE='/content/drive/MyDrive/data/bidmate/code'
if CODE not in sys.path: sys.path.insert(0, CODE)
os.environ['HF_HOME']='/content/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

cfg=types.ModuleType('config')
cfg.ENV='colab'
cfg.PROJECT_ROOT=Path(f'/content/bidmate_{_CHUNK_TAG}')
cfg.DATASET_DIR=cfg.PROJECT_ROOT
cfg.CHUNKS_PATH=cfg.PROJECT_ROOT/_CHUNK_FILE
cfg.CHROMA_PATH=Path(_CHROMA_DIR)
cfg.BM25_PATH=cfg.PROJECT_ROOT/_BM25_FILE
cfg.EVAL_PATH=cfg.PROJECT_ROOT/'eval'
cfg.RESULT_DIR=cfg.PROJECT_ROOT/'eval_results'
cfg.ADAPTER_PATH=Path('/content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter')
cfg.LOG_PATH=cfg.PROJECT_ROOT/'web_user_access.log'
cfg.BASE_MODEL_ID='microsoft/Phi-4-mini-instruct'
cfg.LLM_MODEL='microsoft/Phi-4-mini-instruct'
cfg.EMBED_MODEL_ID='nlpai-lab/KURE-v1'
cfg.RERANKER_ID='BAAI/bge-reranker-v2-m3'
cfg.MAX_TOKENS_REWRITE=300; cfg.MAX_TOKENS_GENERATE=800
cfg.COLLECTION_NAME=_COLLECTION_NAME
cfg.EXPECT_N=_EXPECT_N
cfg.OUT_TAG_DIR=_OUT_TAG_DIR
cfg.AGENCY_KEY=_AGENCY_KEY
cfg.SIG_TH=0.5
cfg.DENSE_K=15; cfg.SPARSE_K=15; cfg.RRF_K=60; cfg.TOP_K=5
cfg.MMR_LAMBDA=0.6; cfg.MMR_TOP_N=20; cfg.RERANK_TOP_N=15; cfg.BATCH_SIZE=64
sys.modules['config']=cfg
assert cfg.EXPECT_N>0, '❌ EXPECT_N=0 — [1] 재실행'
print(f'config OK → tag={_CHUNK_TAG} | col={cfg.COLLECTION_NAME} | expect={cfg.EXPECT_N:,} | agency={cfg.AGENCY_KEY!r} | sig={cfg.SIG_TH}')
print('  CHROMA:', cfg.CHROMA_PATH); print('  BM25:', cfg.BM25_PATH); print('  OUT:', cfg.OUT_TAG_DIR)

In [ ]:
# [3] pre-check — GPU + 파일 존재 + chroma count>0
import torch, pickle, json, os, gc, chromadb
from pathlib import Path
import config as C
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  GPU :', torch.cuda.get_device_name(0))
    print('  VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
for k,p in {'CHUNKS':C.CHUNKS_PATH,'CHROMA':C.CHROMA_PATH,'BM25':C.BM25_PATH,
            'EVAL':C.EVAL_PATH/'eval_retrieval_579.csv','ADAPTER':C.ADAPTER_PATH}.items():
    print(f'{"OK" if Path(p).exists() else "MISSING":8}{k:8}{p}')
with open(C.CHUNKS_PATH, encoding='utf-8') as f: n_chunks=len(json.load(f))
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
n_chroma=chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME).count()
with open(C.BM25_PATH,'rb') as f: n_bm25=len(pickle.load(f)['chunk_ids'])
print(f'\n청크JSON {n_chunks:,} | chroma {n_chroma:,} | bm25 {n_bm25:,} (기준 {C.EXPECT_N:,})')
if Path(C.ADAPTER_PATH).exists(): print('어댑터:', os.listdir(C.ADAPTER_PATH)[:6])
assert n_chroma>0, '❌ chroma 비어있음 — [1] 재확인'
assert n_chroma==C.EXPECT_N, f'chroma 불일치 {n_chroma:,}!={C.EXPECT_N:,}'
if n_bm25!=n_chroma: print(f'⚠️ bm25({n_bm25:,})!=chroma({n_chroma:,}) — 하이브리드 정합 확인')
print('✅ pre-check 통과')

In [ ]:
# [4] 서빙 모듈 로드 + retriever 조립 (새 retrieval.py 내장기능 사용 — 패치 불필요)
import importlib.util, sys, pickle, gc, os
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import config as C

CODE='/content/drive/MyDrive/data/bidmate/code'
def load_module(name):
    path=f'{CODE}/{name}.py'; assert os.path.exists(path), f'파일 없음: {path}'
    spec=importlib.util.spec_from_file_location(name, path)
    mod=importlib.util.module_from_spec(spec); sys.modules[name]=mod
    spec.loader.exec_module(mod); return mod

Rtv=load_module('retrieval')
Rtv.DEVICE=DEVICE
Rtv.SIG_TH=C.SIG_TH                       # D타입 거절 임계값(모듈 전역) 동기화
all_chunks=Rtv.load_chunks()
Rtv.ALL_AGENCIES=list({c['metadata'].get(C.AGENCY_KEY,'') for c in all_chunks
                       if c['metadata'].get(C.AGENCY_KEY,'')})
print(f'retrieval 로드 OK | load_chunks: {len(all_chunks):,} | agencies({C.AGENCY_KEY}): {len(Rtv.ALL_AGENCIES)}')

embed_model=SentenceTransformer(C.EMBED_MODEL_ID, device=DEVICE, cache_folder='/content/hf_cache/hub')
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
collection=chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME)
_cnt=collection.count(); print('chroma count:', f'{_cnt:,}')
assert _cnt>0 and _cnt==C.EXPECT_N, f'❌ chroma count={_cnt:,} (기대 {C.EXPECT_N:,})'

with open(C.BM25_PATH,'rb') as f: bm=pickle.load(f)
reranker=CrossEncoder(C.RERANKER_ID, device=DEVICE)
# ★ agency_meta_key / sig_th 를 생성자에 주입 (인메모리 패치 대체)
retriever=Rtv.BidMateRetriever(
    collection=collection, bm25_index=bm['index'], bm25_chunk_ids=bm['chunk_ids'],
    bm25_texts=bm['texts'], embed_model=embed_model, all_chunks=all_chunks, reranker=reranker,
    agency_meta_key=C.AGENCY_KEY, sig_th=C.SIG_TH,
)
Rtv.retriever=retriever
Rtv.retriever_c=None     # 단일 컬렉션 평가 → C 라우팅 비활성 (get_context 는 history 무관하게 retriever 사용)
print(f'✅ retriever 초기화 완료 (agency_meta_key={C.AGENCY_KEY!r}, sig_th={C.SIG_TH})')

In [ ]:
# [4b] generator 로드 (서빙 generation.py + FT Phi 어댑터)
import torch
Gen=load_module('generation')
generator=Gen.init_generator(Rtv.get_context)
Gen.generator=generator
if torch.cuda.is_available():
    print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,1),
          '/', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
print('✅ generator 초기화 완료 (Phi-4-mini + LoRA)')

In [ ]:
# [4c] 스모크 — 검색 단독 + 메타필터 + 생성 1건 (0점 조기 감지)
import time, pandas as pd, json, ast
eval_df=pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('타입 분포:', eval_df['type'].value_counts().to_dict())
def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x] if isinstance(h,list) else []
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

r=eval_df.iloc[0]; hist=_ph(r.get('history','')); mf=_pm(r.get('metadata_filter',''))
rewritten=generator._rewrite_query(r['question'], hist or None)
rr=retriever.retrieve(rewritten, meta_filter=mf); top=rr.get('top_chunks',[])
print(f"\n[검색] rewritten={rewritten[:50]!r} | top_chunks={len(top)}")
assert len(top)>0 or r['type']=='D', '❌ 검색 0건 — chroma/메타필터/기관키 재확인'
names=[c['metadata'].get('source_file','') for c in top]
print('[검색] retrieved_names:', json.dumps(names, ensure_ascii=False))
if mf:
    where=retriever._build_chroma_where(mf)
    got=collection.get(where=where, limit=3)
    print(f'[필터] where={where} → 매칭 {len(got["ids"])}건', '' if got['ids'] else '⚠️ 0건이면 키/값 불일치')
t=time.time(); out=generator.generate(r['question'], history=hist or None, meta_filter=mf); dt=time.time()-t
print(f'\n[{r["type"]}] {r["question"][:40]}'); print('답변:', out['answer'][:200])
print(f'\n1건 {dt:.1f}초 → 579행 예상 {dt*579/3600:.1f}시간')

In [ ]:
# [5] 579행 생성 — question 기준 done(id 중복 회피), 25행 체크포인트
import pandas as pd, json, ast, time, os
from tqdm.auto import tqdm
import config as _C
OUT=_C.OUT_TAG_DIR; os.makedirs(OUT, exist_ok=True)
GEN_PATH=f'{OUT}/e2e_kure_phi_ft_579.csv'

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x] if isinstance(h,list) else []
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

eval_df=pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('평가셋:', len(eval_df), '| 고유 question:', eval_df['question'].nunique(),
      '| 타입:', eval_df['type'].value_counts().to_dict())

done, records=set(), []
if os.path.exists(GEN_PATH):
    prev=pd.read_csv(GEN_PATH)
    prev=prev[prev['answer'].notna() & (prev['answer'].astype(str).str.len()>0)].drop_duplicates(subset='question')
    records=prev.to_dict('records'); done=set(prev['question'])
    print('체크포인트 재사용:', len(done))

pending=eval_df.drop_duplicates(subset='question')
pending=pending[~pending['question'].isin(done)]
print('신규:', len(pending))

for _, row in tqdm(pending.iterrows(), total=len(pending), desc='생성'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    t0=time.time(); rewritten=generator._rewrite_query(q, hist or None)
    rr=retriever.retrieve(rewritten, meta_filter=mf); retr_ms=round((time.time()-t0)*1000)
    top=rr['top_chunks']
    t1=time.time(); out=generator.generate(q, history=hist or None, meta_filter=mf); gen_ms=round((time.time()-t1)*1000)
    records.append({
        'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'question':q,'rewritten_query':rewritten,
        'ground_truth_answer':row['ground_truth_answer'],'ground_truth_docs':row['ground_truth_docs'],
        'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('source_file','') for c in top], ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms,
    })
    if len(records)%25==0:
        pd.DataFrame(records).to_csv(GEN_PATH, index=False, encoding='utf-8-sig')

gen_df=pd.DataFrame(records).drop_duplicates(subset='question')
gen_df.to_csv(GEN_PATH, index=False, encoding='utf-8-sig')
print('✅ 생성 완료:', len(gen_df), '| 고유 question:', gen_df['question'].nunique())

In [ ]:
# [5b] 생성 결과 무결성 점검 (question 기준)
import pandas as pd, json
import config as _C
GEN_PATH=f'{_C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv'
df=pd.read_csv(GEN_PATH); ev=pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('저장된 행:', len(df), '| 고유 question:', df['question'].nunique())
print('answer 빈 행:', df['answer'].isna().sum()+(df['answer'].astype(str).str.len()==0).sum())
print('eval 에 있는데 저장 안 된 question:', len(set(ev['question'])-set(df['question'])))
print('중복 question:', df['question'].duplicated().sum())
def _empty(raw):
    try: v=json.loads(raw)
    except Exception: return True
    return (not isinstance(v,list)) or len(v)==0 or all(not str(x).strip() for x in v)
n_empty=df['retrieved_names'].apply(_empty).sum()
print(f'retrieved_names 비어있는 행: {n_empty} / {len(df)} (D타입 거절 포함)')
assert n_empty < len(df)*0.5, '❌ 검색결과 절반 이상 비어있음 — chroma/메타필터 재점검'
print('✅ 무결성 통과')

In [ ]:
# [6] Retrieval 지표 — Hit@5/MRR/nDCG
import pandas as pd, json, ast, math, os
import config as _C
OUT=_C.OUT_TAG_DIR
gen_df=pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv')
def _tolist(raw):
    if isinstance(raw,list): return raw
    for fn in (json.loads, ast.literal_eval):
        try:
            v=fn(raw)
            if isinstance(v,list): return v
        except Exception: pass
    return []
def _norm(x): return os.path.splitext(str(x).strip())[0].replace(' ','').lower()
def rmetrics(row, k=5):
    gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
    got=[_norm(x) for x in _tolist(row['retrieved_names'])][:k]
    if not gts: return None
    rank=next((i for i,g in enumerate(got,1) if g in gts), 0)
    dcg=sum(1.0/math.log2(i+1) for i,g in enumerate(got,1) if g in gts)
    idcg=sum(1.0/math.log2(i+1) for i in range(1,min(len(gts),k)+1))
    return pd.Series({'hit@5':1.0 if rank else 0.0,'mrr':1.0/rank if rank else 0.0,'ndcg':dcg/idcg if idcg else 0.0})
rm=gen_df.join(gen_df.apply(rmetrics, axis=1)); valid=rm.dropna(subset=['hit@5'])
print(f'대상 {len(valid)}행')
print('전체:', valid[['hit@5','mrr','ndcg']].mean().round(4).to_dict())
print('\n타입별:\n', valid.groupby('type')[['hit@5','mrr','ndcg']].mean().round(4))
if valid['hit@5'].mean()==0.0:
    print('\n⚠️ Hit@5 전체 0 — 정규화 매칭 진단:')
    for _, row in valid.head(3).iterrows():
        gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
        got=[_norm(x) for x in _tolist(row['retrieved_names'])][:5]
        print('  GT:', gts); print('  GOT:', got); print('  교집합:', set(gts)&set(got),'\n')
s=valid.groupby('type')[['hit@5','mrr','ndcg']].mean(); s.loc['ALL']=valid[['hit@5','mrr','ndcg']].mean()
s.to_csv(f'{OUT}/retrieval_metrics_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [7] Generation Judge (gpt-5.4-mini async, 6지표, 50행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
from openai import AsyncOpenAI
from tqdm.auto import tqdm
import config as _C
nest_asyncio.apply()
OUT=_C.OUT_TAG_DIR
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'),'OPENAI_API_KEY 필요'
_M='gpt-5.4-mini'; _client=AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM=asyncio.Semaphore(15); _RETRY=3
_JP={
'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n"
                "5점: 모든 내용이 Context 근거. 1점: Context 무관/날조.\n[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n"
             "5점: 핵심을 정확·간결히 해결. 1점: 동문서답.\n[Question]\n{query}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n"
             "5점: 근거 없으면 적절히 거절. 1점: 근거 없이 날조.\n[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n"
               "5점: 모두 일치. 1점: 핵심 불일치.\n[Ground Truth]\n{ground_truth}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n"
                     "5점: 모두 필요. 1점: 대부분 불필요.\n[Question]\n{query}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n"
                  "5점: 모든 핵심 포함. 1점: 누락 심각.\n[Ground Truth]\n{ground_truth}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
}
def _parse(raw):
    if not raw: return None
    m=re.search(r'점수\s*:\s*(\d)',raw)
    if m: return int(m.group(1))
    s=raw.strip()
    if s.isdigit() and 1<=int(s)<=5: return int(s)
    d=re.findall(r'\b[1-5]\b',raw); return int(d[0]) if d else None
async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r=await _client.chat.completions.create(model=_M,
                    messages=[{'role':'user','content':prompt}], max_completion_tokens=20, timeout=15)
                return _parse(r.choices[0].message.content)
            except Exception:
                if a==_RETRY-1: return None
                await asyncio.sleep(2**a)
async def score_one(q,ctx,ans,gt=None):
    tasks,none_keys={},[]
    for m in ('faithfulness','relevance','rejection'):
        tasks[m]=_ask(_JP[m].format(context=ctx,query=q,answer=ans))
    for m in ('correctness','context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m]=_ask(_JP[m].format(ground_truth=gt,answer=ans,context=ctx))
        else: none_keys.append(m)
    tasks['context_precision']=_ask(_JP['context_precision'].format(query=q,context=ctx))
    vals=await asyncio.gather(*tasks.values()); res=dict(zip(tasks.keys(),vals))
    for k in none_keys: res[k]=None
    return res
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
JUDGE_PATH=f'{OUT}/quant_scores_kure_phi_ft.csv'
async def run_judge():
    gdf=pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv').drop_duplicates(subset='question')
    done,rows=set(),[]
    if os.path.exists(JUDGE_PATH):
        ck=pd.read_csv(JUDGE_PATH); ck=ck[ck['relevance'].notna()].drop_duplicates(subset='question')
        rows=ck.to_dict('records'); done=set(ck['question']); print('judge 체크포인트:',len(done))
    pending=gdf[~gdf['question'].isin(done)]; print('judge 신규:',len(pending))
    for _,row in tqdm(pending.iterrows(), total=len(pending), desc='judge'):
        ans=row['answer']
        base={'id':row['id'],'question':row['question'],'type':row['type'],'difficulty':row['difficulty']}
        if not isinstance(ans,str) or '오류' in str(ans)[:30]:
            for m in _MET: base[m]=None
        else:
            base.update(await score_one(row['question'],row['retrieved_context'],ans,row.get('ground_truth_answer')))
        rows.append(base)
        if len(rows)%50==0: pd.DataFrame(rows).to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out=pd.DataFrame(rows).drop_duplicates(subset='question')
    out.to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig'); print('✅ judge 완료:',len(out)); return out
judge_df=asyncio.get_event_loop().run_until_complete(run_judge())

In [ ]:
# [8] Generation 요약
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
judge_df=pd.read_csv(f'{OUT}/quant_scores_kure_phi_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s=judge_df.groupby('type')[_MET].mean(); s.loc['ALL']=judge_df[_MET].mean()
s.round(3).to_csv(f'{OUT}/generation_summary_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [9] Release Gate
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
retr=pd.read_csv(f'{OUT}/retrieval_metrics_kure_phi_ft.csv',index_col=0)
genm=pd.read_csv(f'{OUT}/generation_summary_kure_phi_ft.csv',index_col=0)
def v(x,p,g): return 'GOOD' if x>=g else ('PASS' if x>=p else 'FAIL')
print('RETRIEVAL (전체)')
print(f"  Hit@5 {retr.loc['ALL','hit@5']:.3f} → {v(retr.loc['ALL','hit@5'],0.90,0.95)}")
print(f"  MRR   {retr.loc['ALL','mrr']:.3f} → {v(retr.loc['ALL','mrr'],0.82,0.87)}")
print(f"  nDCG  {retr.loc['ALL','ndcg']:.3f} → {v(retr.loc['ALL','ndcg'],0.78,0.83)}")
print('타입별 MRR')
for t,(p,g) in {'A':(0.92,0.95),'B':(0.77,0.82),'C':(0.88,0.93),'D':(0.81,0.86),'E':(0.82,0.87)}.items():
    if t in retr.index: print(f"  {t} {retr.loc[t,'mrr']:.3f} → {v(retr.loc[t,'mrr'],p,g)}")
print('GENERATION (≥3.5 PASS / ≥4.0 GOOD)')
for m in ['faithfulness','relevance','rejection','context_precision']:
    if m in genm.columns:
        print(f"  {m:18} {genm.loc['ALL',m]:.3f} → {v(genm.loc['ALL',m],3.5,4.0)}")

In [ ]:
# [10] 산출물 목록 + Drive 백업
import os, shutil
import config as _C
OUT=_C.OUT_TAG_DIR
print('OUT:', OUT)
for f in sorted(os.listdir(OUT)):
    p=os.path.join(OUT,f)
    if os.path.isfile(p): print(f'  {os.path.getsize(p)/1024:8.1f} KB  {f}')
DST=f'/content/drive/MyDrive/data/bidmate/outputs/{_C.OUT_TAG_DIR.split("/")[-1]}'
os.makedirs(os.path.dirname(DST), exist_ok=True)
if os.path.abspath(OUT)!=os.path.abspath(DST):
    shutil.copytree(OUT, DST, dirs_exist_ok=True)
print('백업:', DST)

In [ ]:
# [11] 정성 분석 — 오류 역추적 / C타입 맥락 / 타입별 요약
import pandas as pd, os
import config as _C
OUT=_C.OUT_TAG_DIR; QUAL_DIR=f'{OUT}/qual'; os.makedirs(QUAL_DIR, exist_ok=True)
gen_df=pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv'); judge_df=pd.read_csv(f'{OUT}/quant_scores_kure_phi_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']; ERROR_TH=3.0
score_cols=['question']+[m for m in _MET if m in judge_df.columns]
gen_df=gen_df.drop_duplicates(subset='question'); judge_df=judge_df.drop_duplicates(subset='question')
merged=gen_df.merge(judge_df[score_cols], on='question', how='left')
print(f'병합: {len(merged)}행')
mask=pd.Series(False, index=merged.index)
for m in ['faithfulness','relevance']:
    if m in merged.columns: mask|=(merged[m].notna() & (merged[m]<=ERROR_TH))
mask|=merged['answer'].astype(str).str.contains('오류', na=False)
err_cols=[c for c in ['id','type','difficulty','question','ground_truth_answer','answer','retrieved_context']+_MET if c in merged.columns]
merged[mask][err_cols].to_csv(f'{QUAL_DIR}/qual_error_analysis.csv', index=False, encoding='utf-8-sig')
print(f'1) 오류 케이스: {int(mask.sum())}건')
kws=['그 ','저 ','위에서','앞서','아까','해당','그것','거기']
cmask=(merged['type']=='C') | merged['question'].astype(str).str.contains('|'.join(kws), na=False, regex=True)
c_cols=[c for c in ['id','type','question','ground_truth_answer','answer','retrieved_context'] if c in merged.columns]
merged[cmask][c_cols].to_csv(f'{QUAL_DIR}/qual_ctype_tracking.csv', index=False, encoding='utf-8-sig')
print(f'2) C타입/맥락: {int(cmask.sum())}건')
rows=[]
for t in ['A','B','C','D','E']:
    sub=merged[merged['type']==t]
    if sub.empty: continue
    r={'type':t,'n':len(sub)}
    for m in _MET: r[m]=round(sub[m].dropna().mean(),3) if m in sub.columns else None
    r['gen_errors']=int(sub['answer'].astype(str).str.contains('오류', na=False).sum()); rows.append(r)
summary_df=pd.DataFrame(rows); summary_df.to_csv(f'{QUAL_DIR}/qual_summary.csv', index=False, encoding='utf-8-sig')
print('3) 타입별 요약\n'); print(summary_df.to_string(index=False))
print(f'\n✅ 정성 분석 저장: {QUAL_DIR}/')